In [ ]:
def main(datasources, start_date, end_date):
      """
      微观市场结构因子模板：
      基于 1 分钟盘口快照计算日频订单簿压力因子。

      返回:
          pd.DataFrame: ['date', 'instrument', 'factor']
      """
      import numpy as np
      import pandas as pd
      import dai

      bar1m = datasources["bar1m"]

      sql = f"""
      WITH minute_feature AS (
          SELECT
              date::DATE::DATETIME AS date,
              instrument,
              date::TIME AS minute_time,

              (
                  CAST(bid_volume1 AS DOUBLE) - CAST(ask_volume1 AS DOUBLE)
              ) / NULLIF(
                  CAST(bid_volume1 AS DOUBLE) + CAST(ask_volume1 AS DOUBLE),
                  0
              ) AS book_pressure,

              (
                  CAST(ask_price1 AS DOUBLE) - CAST(bid_price1 AS DOUBLE)
              ) / NULLIF(
                  (CAST(ask_price1 AS DOUBLE) + CAST(bid_price1 AS DOUBLE)) / 2.0,
                  0
              ) AS rel_spread,

              (
                  CAST(close AS DOUBLE) - CAST(open AS DOUBLE)
              ) / NULLIF(CAST(open AS DOUBLE), 0) AS minute_ret

          FROM {bar1m}
          WHERE
              bid_volume1 IS NOT NULL
              AND ask_volume1 IS NOT NULL
              AND bid_price1 IS NOT NULL
              AND ask_price1 IS NOT NULL
              AND open > 0
      )

      SELECT
          date,
          instrument,

          AVG(book_pressure) AS avg_pressure,

          AVG(
              CASE
                  WHEN minute_time <= TIME '10:00:00'
                  THEN book_pressure
                  ELSE NULL
              END
          ) AS open_pressure,

          AVG(rel_spread) AS avg_spread,

          AVG(book_pressure * minute_ret) AS pressure_ret,

          STDDEV_SAMP(book_pressure) AS pressure_vol,

          COUNT(*) AS n_minutes

      FROM minute_feature
      GROUP BY date, instrument
      ORDER BY date, instrument
      """

      df = dai.query(
          sql,
          filters={"date": [start_date, end_date]},
          compression=True,
      ).df()

      if df.empty:
          return pd.DataFrame(columns=["date", "instrument", "factor"])

      df["date"] = pd.to_datetime(df["date"]).dt.normalize()

      for col in [
          "avg_pressure",
          "open_pressure",
          "avg_spread",
          "pressure_ret",
          "pressure_vol",
          "n_minutes",
      ]:
          df[col] = pd.to_numeric(df[col], errors="coerce")

      # 一个基础版本：
      # 买盘压力越强越好；
      # 价差越大代表流动性越差，扣分；
      # 压力推动当分钟上涨太多，可能已经被价格反映，扣分；
      # 压力波动太大代表盘口不稳定，扣分。
      df["raw_factor"] = (
          0.50 * df["avg_pressure"].fillna(0.0)
          + 0.30 * df["open_pressure"].fillna(0.0)
          - 0.10 * df["avg_spread"].fillna(0.0)
          - 0.05 * df["pressure_ret"].fillna(0.0)
          - 0.05 * df["pressure_vol"].fillna(0.0)
      )

      # 对不完整交易日做轻微惩罚
      df["raw_factor"] *= np.clip(df["n_minutes"].fillna(0) / 240.0, 0.3, 1.0)

      df["raw_factor"] = df["raw_factor"].replace([np.inf, -np.inf], np.nan)

      # 每日横截面去极值 + 标准化
      def normalize(group):
          raw = pd.to_numeric(group["raw_factor"], errors="coerce")
          med = raw.median()
          mad = (raw - med).abs().median()

          if pd.isna(mad) or mad == 0:
              mad = 1e-8

          clipped = raw.clip(med - 5 * mad, med + 5 * mad)

          std = clipped.std()
          if pd.isna(std) or std == 0:
              std = 1e-8

          group["factor"] = (clipped - clipped.mean()) / std
          return group

      df = df.groupby("date", group_keys=False).apply(normalize)
      df = df[["date", "instrument", "factor"]]
      df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["factor"])

      # 对齐比赛股票池
      stk_pool = dai.query(
          "SELECT date, instrument FROM bigalpha_2026_instruments",
          filters={"date": [start_date, end_date]},
      ).df()
      stk_pool["date"] = pd.to_datetime(stk_pool["date"]).dt.normalize()

      df = pd.merge(df, stk_pool, how="inner", on=["date", "instrument"])

      return (
          df[["date", "instrument", "factor"]]
          .sort_values(["date", "instrument"])
          .reset_index(drop=True)
      )